# SABIO-RK Reaction 618 beta-glucosidase pilot

This is a first curated kinetic-record pilot, not a validated full fungal degradation model.

In [ ]:
from pathlib import Path
import sys

ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path(".").resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from fungal_model.registry import load_registry

registry = load_registry(ROOT / "data_registry" / "registry_index.yml")
registry.registry_id

In [ ]:
from fungal_model.data import load_kinetic_record

record = load_kinetic_record(
    ROOT / "data" / "kinetic_records" / "sabiork" / "case_001_reaction_618_beta_glucosidase" / "curated" / "kinetic_record.yml"
)
record.to_dict()

In [ ]:
from fungal_model.screening import assess_modelability

FUNGUS_ID = "sabiork_beta_glucosidase_source"
SUBSTRATE_ID = "cellobiose"
ENVIRONMENT_ID = "sabiork_reaction_618_selected_conditions"

report = assess_modelability(
    fungus_id=FUNGUS_ID,
    substrate_id=SUBSTRATE_ID,
    environment_id=ENVIRONMENT_ID,
    registry=registry,
    mode="scientific",
)

print(report.summary())
report.to_dict()

In [ ]:
from fungal_model.screening import build_model_config_from_registry_case
from fungal_model.workflows import run_configured_model
import yaml

if report.status == "modelable":
    output_dir = ROOT / "outputs" / "sabiork_reaction_618_beta_glucosidase"

    config = build_model_config_from_registry_case(
        fungus_id=FUNGUS_ID,
        substrate_id=SUBSTRATE_ID,
        environment_id=ENVIRONMENT_ID,
        registry=registry,
        mode="scientific",
        output_directory=str(output_dir),
    )

    output_dir.mkdir(parents=True, exist_ok=True)
    config_path = output_dir / "model_config.yml"
    config_path.write_text(yaml.safe_dump(config.to_dict(), sort_keys=False), encoding="utf-8")

    result = run_configured_model(config_path, output_dir=output_dir / "bundle")
    result.plot_states(output_dir / "states.png")
else:
    print(report.to_dict())

## Exploratory Screen

Homogeneous Michaelis-Menten exploratory ensembles are not implemented in this milestone. Range and distribution parameters are intentionally refused by the deterministic builder.

## Limitations

- enzyme-only kinetic pilot;
- not whole fungus;
- no secretion;
- no uptake;
- no biomass growth;
- no validation against time-course observations;
- parameters only valid for selected SABIO-RK conditions.